# TPU probe — the number that decides whether we train a member of our own

Two days went into porting other people's weights because GPU training is arithmetically impossible:
0.303 passages/s on one T4 is 211 hours per epoch of the 228k pool, against a 30-hour weekly quota. The
**20-hour weekly TPU quota has sat unused the whole time**, because nobody measured it. That was the
mistake.

This measures it on a TPU v3-8, on **one core**, with everything the plan's step 3 needs:

- Qwen3-8B-Base, frozen, bf16
- LoRA (rank 16, alpha 32) on q/k/v/o of the last 8 layers
- a **fixed** sequence length — XLA recompiles on every shape change, so shapes must never vary
- 20 steps at 4 x 512 tokens, timed after 3 warmup steps
- data-parallel across 8 cores is near-linear for compute-bound training, so the projection is x8

If the projected epoch is single-digit hours, we train a weak-class specialist this week. If it is 40, the
plan's step 3 is closed and we say so plainly instead of starting a run that cannot finish.

In [ ]:
"""TPU training probe — the number that decides everything.

We have spent two days porting other people's weights because GPU training is arithmetically impossible:
0.303 passages/s on one T4 means 211 hours for one epoch of the 228k pool, against a 30-hour weekly quota.
The TPU quota (20 h/week) has sat unused the whole time because nobody measured it.

This measures the real number on a TPU v3-8, on one core, with everything the plan needs:

  * Qwen3-8B-Base, frozen, bf16
  * LoRA on the attention projections only (a few hundred thousand trainable parameters)
  * a fixed sequence length (XLA recompiles on every shape change, so shapes must never vary)
  * sequence packing: several short passages per window, so padding is not thrown away
  * 20 steps at bs 4 x 512 tokens, timed after warmup

Then the projection: data-parallel across 8 cores is near-linear for compute-bound training, so
epoch hours ~= (228,000 / passages_per_s) / 3600 / 8. If that number is single-digit, we can train a
member this week. If it is 40, the plan's step 3 is closed and we say so plainly.
"""
import json
import time

import numpy as np
import torch
import torch.nn as nn

T0 = time.time()
print("=== environment ===", flush=True)
try:
    import torch_xla
    import torch_xla.core.xla_model as xm
    print("torch_xla", torch_xla.__version__, flush=True)
    DEVICE = xm.xla_device()
    print("device:", DEVICE, flush=True)
    ON_TPU = True
except Exception as e:
    print(f"torch_xla unavailable ({type(e).__name__}: {e}) — falling back to CPU for a shape-only check",
          flush=True)
    DEVICE = torch.device("cpu")
    ON_TPU = False

SEQ = 512
BS = 4
STEPS = 20
WARMUP = 3
MODEL_DIR = None
import glob
for cfg in sorted(glob.glob("/kaggle/input/**/config.json", recursive=True)):
    try:
        c = json.load(open(cfg))
    except Exception:
        continue
    if c.get("model_type") == "qwen3" and c.get("hidden_size") == 4096:
        MODEL_DIR = cfg.rsplit("/", 1)[0]
        break
print("base model:", MODEL_DIR, flush=True)

## Load the base and bolt on LoRA

In [ ]:
from transformers import AutoConfig, AutoModel, AutoTokenizer

mc = AutoConfig.from_pretrained(MODEL_DIR)
mc.use_cache = False
dtype = torch.bfloat16
model = AutoModel.from_pretrained(MODEL_DIR, config=mc, dtype=dtype, low_cpu_mem_usage=True)
model.to(DEVICE)
model.eval()

# LoRA on the attention projections of the last 8 layers, everything else frozen
R, ALPHA = 16, 32
trainable = []
for p in model.parameters():
    p.requires_grad_(False)


class LoRALinear(nn.Module):
    def __init__(self, base, r, alpha):
        super().__init__()
        self.base = base
        self.A = nn.Parameter(torch.zeros(r, base.in_features, dtype=dtype))
        self.B = nn.Parameter(torch.zeros(base.out_features, r, dtype=dtype))
        nn.init.kaiming_uniform_(self.A, a=5 ** 0.5)
        nn.init.zeros_(self.B)
        self.scale = alpha / r

    def forward(self, x):
        return self.base(x) + (x @ self.A.T @ self.B.T) * self.scale


n_layers = len(model.layers)
for li in range(max(0, n_layers - 8), n_layers):
    lay = model.layers[li]
    for name in ("q_proj", "k_proj", "v_proj", "o_proj"):
        lin = getattr(lay.self_attn, name)
        setattr(lay.self_attn, name, LoRALinear(lin, R, ALPHA))

head = nn.Linear(model.config.hidden_size, 18, dtype=dtype).to(DEVICE)
trainable = [p for p in model.parameters() if p.requires_grad] + list(head.parameters())
print(f"trainable tensors: {len(trainable)} | params: {sum(p.numel() for p in trainable):,}", flush=True)

## Time it

In [ ]:
"""Time it. Fixed shapes only — one shape for every step, or XLA recompiles each time."""
import time

opt = torch.optim.AdamW(trainable, lr=1e-4)
rng = np.random.default_rng(0)
V = model.config.vocab_size

ids = torch.from_numpy(rng.integers(0, min(V, 150000), size=(BS, SEQ)).astype(np.int64)).to(DEVICE)
mask = torch.ones((BS, SEQ), dtype=torch.long, device=DEVICE)
labels = torch.from_numpy(rng.integers(0, 18, size=(BS,)).astype(np.int64)).to(DEVICE)

def one_step():
    opt.zero_grad(set_to_none=True)
    out = model(input_ids=ids, attention_mask=mask, use_cache=False)
    h = out.last_hidden_state
    m = mask.unsqueeze(-1).float()
    pooled = (h.float() * m).sum(1) / m.sum(1).clamp(min=1)
    loss = torch.nn.functional.cross_entropy(head(pooled), labels)
    loss.backward()
    if ON_TPU:
        xm.optimizer_step(opt)
    else:
        opt.step()
    return float(loss.detach())

for i in range(WARMUP):
    one_step()
    if ON_TPU:
        xm.mark_step()
print("warmup done", flush=True)

t0 = time.time()
for i in range(STEPS):
    one_step()
    if ON_TPU:
        xm.mark_step()
dt = time.time() - t0

tokens = BS * SEQ * STEPS
passages = BS * STEPS
tok_s = tokens / dt
ps_s = passages / dt
print("\n=== RESULT ===", flush=True)
print(f"device: {'TPU v3-8 core' if ON_TPU else 'CPU (shape check only)'}", flush=True)
print(f"{STEPS} steps of {BS} x {SEQ} tokens in {dt:.1f}s", flush=True)
print(f"tokens/s (1 core): {tok_s:,.0f}", flush=True)
print(f"passages/s (1 core): {ps_s:.3f}", flush=True)
print(f"projected 8-core passages/s: {ps_s * 8:.2f}", flush=True)
for n_rows, label in ((228_756, "full pool"), (60_000, "weak-class subset")):
    hours_1 = n_rows / ps_s / 3600
    hours_8 = n_rows / (ps_s * 8) / 3600
    print(f"  {label} ({n_rows:,} rows): 1 epoch = {hours_1:.1f} h on 1 core, {hours_8:.1f} h on 8 cores",
          flush=True)
res = {"on_tpu": bool(ON_TPU), "tokens_per_s_1core": round(tok_s, 1),
       "passages_per_s_1core": round(ps_s, 3), "passages_per_s_8core": round(ps_s * 8, 2),
       "epoch_hours_8core_full_pool": round(228_756 / (ps_s * 8) / 3600, 2),
       "epoch_hours_8core_60k": round(60_000 / (ps_s * 8) / 3600, 2),
       "seconds": round(time.time() - T0, 1)}
json.dump(res, open("tpu_probe.json", "w"), indent=2)
print("\nwrote tpu_probe.json", flush=True)